In [ ]:
import json
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import re
import time
from google.colab import userdata

# --- 1. SECURE CONFIGURATION ---
try:
    # Try to get key from Colab Secrets (Left Sidebar -> Key Icon)
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except:
    # Fallback: Ask user to paste it if not found in secrets
    GEMINI_API_KEY = input("Paste your Gemini API Key here: ").strip()

# Your Interest Profile
USER_INTERESTS = "Unexplained Mysteries, Historical Curiosities, Scientific Facts, True Crime, Weird News, Lendas Urbanas"

# The Curiosities Feed List
# --- LISTA ATUALIZADA E COMPLETA ---
RSS_FEEDS = [
    # --- REDDIT (Inglês - Melhores para Shorts Visuais) ---
    "https://www.reddit.com/r/Damnthatsinteresting/top/.rss?t=day",
    "https://www.reddit.com/r/todayilearned/top/.rss?t=day",
    "https://www.reddit.com/r/ExplainLikeImFive/top/.rss?t=day",
    "https://www.reddit.com/r/Creepy/top/.rss?t=day",
    "https://www.reddit.com/r/OddlyTerrifying/top/.rss?t=day",
    "https://www.reddit.com/r/BeAmazed/top/.rss?t=day",

    # --- CURIOSIDADES EM INGLÊS ---
    "https://listverse.com/feed/",
    "https://www.boredpanda.com/feed/",
    "https://www.thefactsite.com/feed/",
    "https://www.mentalfloss.com/rss.xml",
    "https://www.smithsonianmag.com/rss/smart-news/",
    "https://www.livescience.com/feeds/all",
    "https://www.damninteresting.com/feed/",

    # --- BRASILEIROS (Curiosidades, História e Ciência) ---
    "https://super.abril.com.br/feed/",
    "https://super.abril.com.br/mundo-estranho/feed/",
    "https://gizmodo.uol.com.br/feed/",
    "https://aventurasnahistoria.com.br/feed/",

    # SUBSTITUTO: Canaltech Ciência (Excelente para Espaço/Astronomia/Descobertas)
    "https://canaltech.com.br/rss/ciencia/",
]

# --- 2. SCRAPING FUNCTIONS ---
def fetch_rss_items(url):
    print(f"Fetching {url}...")
    items = []
    # Disguise as Chrome to bypass Reddit blocking
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Referer': 'https://www.google.com/', # Finge que veio do Google
        'Upgrade-Insecure-Requests': '1',
        'Connection': 'keep-alive'
    }

    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=10) as response:
            try:
                # Parse XML
                root = ET.fromstring(response.read())
                # Find all items (RSS) or entries (Atom)
                entries = root.findall('.//item') + root.findall('.//{http://www.w3.org/2005/Atom}entry')

                # Fetch slightly more items per feed to ensure we have a good pool for the Top 20
                for item in entries[:15]:
                    title = item.find('title').text if item.find('title') is not None else "No Title"
                    link_obj = item.find('link')
                    # Handle different link formats (RSS vs Atom)
                    if link_obj is not None:
                        link = link_obj.text if link_obj.text else link_obj.get('href')
                    else:
                        link = "#"
                    items.append({"title": title, "link": link})
            except:
                print(f"  -> XML Parse failed for {url} (Likely not a valid RSS feed)")
    except Exception as e:
        print(f"  -> Connection failed: {e}")

    return items

# --- 3. AI ANALYSIS FUNCTION ---
def rank_articles_with_gemini(articles):
    # Prepare list for AI
    titles_list = "\n".join([f"{i}. {a['title']}" for i, a in enumerate(articles)])

    prompt = f"""
    You are a content curator. Rate these articles from 0 to 100 based on: {USER_INTERESTS}.
    Return ONLY a JSON object in this exact format:
    {{ "scores": [ {{ "index": 0, "score": 85 }}, {{ "index": 1, "score": 10 }} ] }}

    ARTICLES:
    {titles_list}
    """

    # USING THE MODEL IDENTIFIED BY DIAGNOSTIC
    model_name = "gemini-2.5-flash"
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={GEMINI_API_KEY}"

    headers = {"Content-Type": "application/json"}
    data = {"contents": [{"parts": [{"text": prompt}]}]}

    try:
        req = urllib.request.Request(url, data=json.dumps(data).encode('utf-8'), headers=headers)
        with urllib.request.urlopen(req) as response:
            result = json.loads(response.read().decode('utf-8'))

            if 'candidates' in result and result['candidates']:
                raw_text = result['candidates'][0]['content']['parts'][0]['text']

                # Clean Markdown (Gemini often wraps JSON in ```json ... ```)
                clean_text = re.sub(r"```json|```", "", raw_text).strip()

                # Extract JSON object safely
                json_start = clean_text.find('{')
                json_end = clean_text.rfind('}') + 1
                if json_start != -1:
                    clean_text = clean_text[json_start:json_end]

                return json.loads(clean_text).get('scores', [])
            return []

    except urllib.error.HTTPError as e:
        print(f"\nAPI ERROR: {e.code} - {e.reason}")
        print(f"URL: {url}")
        return []
    except Exception as e:
        print(f"General Error: {e}")
        return []

# --- 4. MAIN EXECUTION ---
print("--- STARTING AGENT ---")
all_articles = []

# Fetch all feeds
for feed in RSS_FEEDS:
    all_articles.extend(fetch_rss_items(feed))
    time.sleep(0.5) # Be polite to servers

print(f"\nCollected {len(all_articles)} articles.")

if all_articles:
    print("Asking AI (gemini-2.5-flash) to rank them...")

    # Send ALL collected articles to the AI for ranking
    scores = rank_articles_with_gemini(all_articles)

    # Merge scores back into articles
    ranked = []
    for s in scores:
        if s['index'] < len(all_articles):
            art = all_articles[s['index']]
            art['score'] = s['score']
            ranked.append(art)

    # Sort by Score (Descending: High score to Low score)
    ranked.sort(key=lambda x: x.get('score', 0), reverse=True)

    # Print Result: Top 20
    print("\n" + "="*40)
    print(f"TOP 20 CURIOSITIES FOR TODAY")
    print("="*40)
    for i, art in enumerate(ranked[:20], 1):
        print(f"{i}. [{art.get('score',0)}] {art['title']}")
        print(f"   {art['link']}\n")
else:
    print("No articles found.")